### SimpleWebPageReader

웹 페이지를 LlamaIndex Document로 읽어오는 가장 단순한 예제입니다.
네트워크 상태와 대상 사이트의 차단 정책에 따라 실행 결과가 달라질 수 있습니다.

<!-- 학습 보강 셀 -->

## 웹 데이터 로딩의 특징

웹 페이지는 파일과 달리 네트워크 상태, robots 정책, 동적 렌더링, HTML 구조 변화의 영향을 받습니다.
따라서 같은 코드도 실행 시점이나 사이트 정책에 따라 결과가 달라질 수 있습니다.
학습 단계에서는 안정적인 정적 페이지로 먼저 흐름을 익히는 것이 좋습니다.

In [1]:
# 웹 페이지 리더 설치
# !pip install llama-index-readers-web

In [2]:
# SimpleWebPageReader는 URL 목록을 받아 각 페이지의 텍스트를 Document로 변환합니다.
from llama_index.readers.web import SimpleWebPageReader

In [3]:
# 예제 URL 로드
# - 동적 렌더링/차단 정책이 있는 사이트는 본문 추출이 제한될 수 있어 정적 페이지를 사용합니다.
documents = SimpleWebPageReader().load_data(urls=['https://www.example.com'])
print('읽어온 문서 수:', len(documents))
print(documents[0].text[:500])

읽어온 문서 수: 1
<!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more<


<!-- 학습 보강 셀 -->

## 웹 페이지 로드 결과를 확인하는 방법

웹 로더가 성공했다고 해서 항상 유용한 본문이 들어온 것은 아닙니다.
광고, 메뉴, 푸터, 빈 텍스트가 함께 들어올 수 있으므로 `documents[0].text[:500]`처럼 앞부분을 직접 확인해야 합니다.

### Wikipedia Reader

<!-- 학습 보강 셀 -->

## 외부 지식 소스와 metadata

Wikipedia처럼 출처가 명확한 외부 데이터를 가져올 때는 title, url, source를 metadata에 넣어 두는 것이 좋습니다.
나중에 답변 근거를 사용자에게 보여주거나, 특정 출처만 필터링할 때 활용할 수 있습니다.

In [4]:
# Wikipedia 예제에 필요한 패키지 설치
# - Wikimedia API는 User-Agent가 없는 요청을 차단할 수 있으므로 requests로 직접 호출합니다.
# !pip install requests llama-index

In [5]:
import requests
from llama_index.core import Document

In [6]:
# Wikipedia 문서를 직접 가져와 LlamaIndex Document로 변환합니다.
# - Wikimedia API는 User-Agent가 없는 요청을 403으로 차단할 수 있습니다.
WIKIPEDIA_API_URL = 'https://en.wikipedia.org/w/api.php'
USER_AGENT = 'RAG-LlamaIndex-Notebook/1.0 (educational example)'

params = {
    'action': 'query',
    'format': 'json',
    'titles': 'Python (programming language)',
    'prop': 'extracts|info',
    'explaintext': 1,
    'inprop': 'url',
    'redirects': 1,
}

response = requests.get(
    WIKIPEDIA_API_URL,
    params=params,
    headers={'User-Agent': USER_AGENT},
    timeout=10,
)
response.raise_for_status()
data = response.json()
page = next(iter(data['query']['pages'].values()))

documents = [
    Document(
        text=page['extract'],
        metadata={
            'title': page['title'],
            'url': page['fullurl'],
            'source': 'wikipedia',
        },
    )
]

print(documents[0].metadata)
print(documents[0].text[:1000])

{'title': 'Python (programming language)', 'url': 'https://en.wikipedia.org/wiki/Python_(programming_language)', 'source': 'wikipedia'}
Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, "plain English" naming, an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5, capabilities and keywords for typing were added to the language, allowing optional static typing. As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the project's annual releas